In [30]:
import glob
import os
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import boto3
import matplotlib.pyplot as plt
import fabio
from tqdm import tqdm
import h5pyd
import zarr
from zarr.storage import FsspecStore
from numcodecs import Blosc
from zarr.codecs.gzip import GzipCodec
from zarr.codecs import BloscCodec, BloscCname
import fsspec
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow.fs import S3FileSystem
import tiledb
import random
import time
import datetime
import json

# Intro

In [ ]:
os.environ['AWS_ACCESS_KEY_ID'] = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''
os.environ['BUCKET_NAME'] = 'scatterin-thesis'
os.environ['AWS_REGION'] = 'eu-north-1'
os.environ['AWS_S3_GATEWAY'] = 'http://s3.amazonaws.com'

os.environ['SN_PORT'] = '5101'
os.environ['HSDS_ENDPOINT'] = 'http://localhost:5101'
os.environ['HS_ENDPOINT'] = 'http://localhost:5101'
os.environ['LOG_LEVEL'] = 'INFO'
os.environ['HS_USERNAME'] = 'test_user1'
os.environ['HS_PASSWORD'] = 'test'

aws_opts = {
    "key":    os.environ["AWS_ACCESS_KEY_ID"],
    "secret": os.environ["AWS_SECRET_ACCESS_KEY"],
    "client_kwargs": {"region_name": os.environ["AWS_REGION"]},
}

In [32]:
WARMUP = 10
REPS = 50
SEED = 979969114

In [33]:
random.seed(SEED)
np.random.seed(SEED)

In [34]:
indexes = random.sample(range(0, 1295), WARMUP + REPS)

In [85]:
import boto3, datetime as dt

FILTERID = "root"
REGION   = "eu-north-1"

cw = boto3.client("cloudwatch", region_name=REGION)

end   = dt.datetime.now(dt.timezone.utc).replace(second=0, microsecond=0)
start = end - dt.timedelta(days=2)

start = int(dt.datetime(2025, 9, 15, 15, 17).timestamp())
end   = int(end.timestamp())

def q(metric, stat):
    return {
        "Id": f"{metric.lower()}_{stat.lower()}",
        "MetricStat": {
            "Metric": {
                "Namespace": "AWS/S3",
                "MetricName": metric,
                "Dimensions": [
                    {"Name":"BucketName","Value":"scatterin-thesis"},
                    {"Name":"FilterId","Value":FILTERID},
                ],
            },
            "Period": 600000,
            "Stat": stat,
        },
        "ReturnData": True,
    }

resp = cw.get_metric_data(
    MetricDataQueries=[
        q("GetRequests", "Sum"),
        q("BytesDownloaded", "Sum"),
        q("FirstByteLatency", "p95"),
        q("TotalRequestLatency", "p95"),
        q("FirstByteLatency", "Average"),
        q("TotalRequestLatency", "Average"),
    ],
    StartTime=start,
    EndTime=end,
    ScanBy="TimestampAscending",
    MaxDatapoints=100000,
)

for r in resp["MetricDataResults"]:
    print(r["Label"], len(r["Timestamps"]))
    for ts, val in zip(r["Timestamps"], r["Values"]):
        print(" ", ts, val)


GetRequests 1
  2025-09-15 15:17:00+00:00 3888.0
BytesDownloaded 1
  2025-09-15 15:17:00+00:00 66074950267.0
FirstByteLatency p95 1
  2025-09-15 15:17:00+00:00 79.22124638164908
TotalRequestLatency p95 1
  2025-09-15 15:17:00+00:00 272.6958069276128
FirstByteLatency Average 1
  2025-09-15 15:17:00+00:00 60.78960905349794
TotalRequestLatency Average 1
  2025-09-15 15:17:00+00:00 205.66975308641975


In [86]:
data_type = input("Enter type (zarr/tiledb/hdf5/root): ").strip().lower()
compression = input("Enter compression (gzip/lz4/zstd): ").strip().lower()
size_type = input("Enter type size (full/slice): ").strip().lower()

filename = f"{data_type}_{compression}_{size_type}_cloudwatch.json"
out_path = os.path.join("logs", filename)

if os.path.exists(out_path):
    print(f"File {out_path} already exists. Aborting.")
else:
    out = {}
    for r in resp["MetricDataResults"]:
        label = r["Label"].replace(" ", "")
        if label == "GetRequests":
            key = "GetRequestsSum"
        elif label == "BytesDownloaded":
            key = "BytesDownloadedSum"
        elif "p95" in label:
            key = label.replace("p95", "P95")
        elif "Average" in label:
            key = label.replace("Average", "Average")
        else:
            key = label
        out[key] = sum(r["Values"])

    os.makedirs("logs", exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(out, f, indent=4)
    print(f"Saved to {out_path}")

Saved to logs/root_zstd_full_cloudwatch.json


# Zarr

### LZ4

In [44]:
store_url_lz4 = f"s3://scatterin-thesis/zarr/lz4.zarr"
store_lz4 = FsspecStore.from_url(store_url_lz4, storage_options=aws_opts)
root_lz4 = zarr.open_group(store=store_lz4, mode="r")

In [45]:
arr = root_lz4.get("data")
dtype = np.dtype(arr.dtype)
cz, cy, cx = arr.chunks
chunk_bytes = cz * cy * cx * dtype.itemsize

In [46]:
def read_window(win):
    t0 = time.perf_counter_ns()
    data = arr[win, : :]
    t1 = time.perf_counter_ns()
    return t1 - t0

In [ ]:
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

for w in indexes[:WARMUP]:
    _ = read_window(w)

rows = []
for i, w in enumerate(indexes[WARMUP:WARMUP+REPS], 1):
    t = read_window(w)
    rows.append({
        "i": i,
        "t_total_ns": t,
    })

out = f"logs/zarr_lz4_slice_{RUN_ID}.json"

with open(out, "w") as f:
    json.dump({
        "run_id": RUN_ID,
        "data": rows
    }, f)

### FULLSCAN

In [76]:
def read_full():
    t0 = time.perf_counter_ns()
    data = arr[:, : :]
    t1 = time.perf_counter_ns()
    return t1 - t0

In [ ]:
RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

_ = read_full()

rows = []
for i in tqdm(range(1, REPS + 1)):
    t = read_full()
    rows.append({
        "i": i,
        "t_total_ns": t,
    })

out = f"logs/zarr_lz4_full_{RUN_ID}.json"

with open(out, "w") as f:
    json.dump({
        "run_id": RUN_ID,
        "data": rows
    }, f)

100%|██████████| 50/50 [24:36<00:00, 29.52s/it]


## GZIP

In [6]:
store_url_gzip = f"s3://scatterin-thesis/zarr/gzip.zarr"
store_gzip = FsspecStore.from_url(store_url_gzip, storage_options=aws_opts)
root_gzip = zarr.open_group(store=store_gzip, mode="r")

arr = root_gzip.get("data")
dtype = np.dtype(arr.dtype)
cz, cy, cx = arr.chunks
chunk_bytes = cz * cy * cx * dtype.itemsize

### Slice

In [7]:
def read_window(win):
    t0 = time.perf_counter_ns()
    data = arr[win, : :]
    t1 = time.perf_counter_ns()
    return t1 - t0

RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

for w in indexes[:WARMUP]:
    _ = read_window(w)

rows = []
for i, w in enumerate(indexes[WARMUP:WARMUP+REPS], 1):
    t = read_window(w)
    rows.append({
        "i": i,
        "t_total_ns": t,
    })

out = f"logs/zarr_gzip_slice_{RUN_ID}.json"

with open(out, "w") as f:
    json.dump({
        "run_id": RUN_ID,
        "data": rows
    }, f)

### Full Scan

In [17]:
def read_full():
    t0 = time.perf_counter_ns()
    data = arr[:, : :]
    t1 = time.perf_counter_ns()
    return t1 - t0


_ = read_full()

rows = []
for i in tqdm(range(1, REPS + 1)):
    t = read_full()
    rows.append({
        "i": i,
        "t_total_ns": t,
    })

os.makedirs("logs", exist_ok=True)
out = f"logs/zarr_gzip_full_{RUN_ID}.json"

with open(out, "w") as f:
    json.dump({
        "run_id": RUN_ID,
        "data": rows
    }, f)

100%|██████████| 50/50 [29:41<00:00, 35.64s/it]


### ZSTD

In [21]:
store_url_zstd = f"s3://scatterin-thesis/zarr/zstd.zarr"
store_zstd = FsspecStore.from_url(store_url_zstd, storage_options=aws_opts)
root_zstd = zarr.open_group(store=store_zstd, mode="r")

arr = root_zstd.get("data")
dtype = np.dtype(arr.dtype)
cz, cy, cx = arr.chunks
chunk_bytes = cz * cy * cx * dtype.itemsize

### Slice

In [22]:
def read_window(win):
    t0 = time.perf_counter_ns()
    data = arr[win, : :]
    t1 = time.perf_counter_ns()
    return t1 - t0

RUN_ID = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

for w in indexes[:WARMUP]:
    _ = read_window(w)

rows = []
for i, w in enumerate(indexes[WARMUP:WARMUP+REPS], 1):
    t = read_window(w)
    rows.append({
        "i": i,
        "t_total_ns": t,
    })

out = f"logs/zarr_zstd_slice_{RUN_ID}.json"

with open(out, "w") as f:
    json.dump({
        "run_id": RUN_ID,
        "data": rows
    }, f)

In [25]:
def read_full():
    t0 = time.perf_counter_ns()
    data = arr[:, : :]
    t1 = time.perf_counter_ns()
    return t1 - t0


_ = read_full()

rows = []
for i in tqdm(range(1, REPS + 1)):
    t = read_full()
    rows.append({
        "i": i,
        "t_total_ns": t,
    })

os.makedirs("logs", exist_ok=True)
out = f"logs/zarr_zstd_full_{RUN_ID}.json"

with open(out, "w") as f:
    json.dump({
        "run_id": RUN_ID,
        "data": rows
    }, f)

100%|██████████| 50/50 [25:20<00:00, 30.40s/it]


# TileDB

### intro

In [7]:
cfg = {
    "vfs.s3.region":                 os.environ["AWS_REGION"],
    "vfs.s3.aws_access_key_id":      os.environ["AWS_ACCESS_KEY_ID"],
    "vfs.s3.aws_secret_access_key":  os.environ["AWS_SECRET_ACCESS_KEY"],
    "vfs.s3.scheme":                 "https",
    "vfs.s3.use_virtual_addressing": "true",
}
ctx = tiledb.Ctx(tiledb.Config(cfg))
os.makedirs("logs", exist_ok=True)

def _idxs(n: int) -> list[int]:
    random.seed(SEED); np.random.seed(SEED)
    need = WARMUP + REPS
    if need <= n:
        return random.sample(range(n), need)
    return [random.randrange(n) for _ in range(need)]

def run_slice(uri: str, label: str) -> str:
    with tiledb.DenseArray(uri, mode="r", ctx=ctx) as A:
        n, _, _ = A.shape
        idxs = _idxs(n)

        # warmup not recorded
        for i in idxs[:WARMUP]:
            _ = A[i, :, :]

        rows = []
        for j, i in enumerate(idxs[WARMUP:WARMUP+REPS], 1):
            t0 = time.perf_counter_ns()
            _ = A[i, :, :]
            t1 = time.perf_counter_ns()
            rows.append({"i": j, "t_total_ns": int(t1 - t0)})

    run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    out = f"logs/tiledb_{label}_slice_{run_id}.json"
    with open(out, "w") as f:
        json.dump({"run_id": run_id, "data": rows}, f)
    print(out)
    return out

def run_full(uri: str, label: str) -> str:
    with tiledb.DenseArray(uri, mode="r", ctx=ctx) as A:
        def read_all() -> int:
            t0 = time.perf_counter_ns()
            _ = A[:, :, :]
            t1 = time.perf_counter_ns()
            return t1 - t0

        # single warmup, not recorded
        _ = read_all()

        rows = []
        for j in tqdm(range(1, REPS + 1)):
            dt = read_all()
            rows.append({"i": j, "t_total_ns": int(dt)})

    run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    out = f"logs/tiledb_{label}_full_{run_id}.json"
    with open(out, "w") as f:
        json.dump({"run_id": run_id, "data": rows}, f)
    print(out)
    return out

URIS = {
    "gzip": "s3://scatterin-thesis/tiledb/gzip.tdb",
    "lz4":  "s3://scatterin-thesis/tiledb/lz4.tdb",
    "zstd": "s3://scatterin-thesis/tiledb/zstd.tdb",
}

### GZIP

In [20]:
run_slice(URIS["gzip"], "gzip")

logs/tiledb_gzip_slice_20250914-062234.json


'logs/tiledb_gzip_slice_20250914-062234.json'

In [33]:
run_full(URIS["gzip"], "gzip")

100%|██████████| 50/50 [09:23<00:00, 11.27s/it]

logs/tiledb_gzip_full_20250914-063913.json


'logs/tiledb_gzip_full_20250914-063913.json'

### LZ4

In [59]:
run_slice(URIS["lz4"], "lz4")

logs/tiledb_lz4_slice_20250914-064937.json


'logs/tiledb_lz4_slice_20250914-064937.json'

In [76]:
run_full(URIS["lz4"], "lz4")

100%|██████████| 50/50 [11:20<00:00, 13.61s/it]

logs/tiledb_lz4_full_20250914-070640.json


'logs/tiledb_lz4_full_20250914-070640.json'

### ZSTD

In [105]:
run_slice(URIS["zstd"], "zstd")

logs/tiledb_zstd_slice_20250914-071737.json


'logs/tiledb_zstd_slice_20250914-071737.json'

In [122]:
run_full(URIS["zstd"], "zstd")

100%|██████████| 50/50 [08:40<00:00, 10.40s/it]

logs/tiledb_zstd_full_20250914-073059.json


'logs/tiledb_zstd_full_20250914-073059.json'

# HSDS

### Intro

In [ ]:
HS_ENDPOINT = os.getenv("HS_ENDPOINT", "http://localhost:5101")
HS_USERNAME = os.getenv("HS_USERNAME", "test_user1")
HS_PASSWORD = os.getenv("HS_PASSWORD", "test")

os.makedirs("logs", exist_ok=True)

def _idxs(n: int) -> list[int]:
    random.seed(SEED); np.random.seed(SEED)
    need = WARMUP + REPS
    if n <= 0:
        raise ValueError("Empty dataset")
    if need <= n:
        return random.sample(range(n), need)
    return [random.randrange(n) for _ in range(need)]

def run_slice(h5_path: str, label: str) -> str:
    with h5pyd.File(h5_path, "r", endpoint=HS_ENDPOINT, username=HS_USERNAME, password=HS_PASSWORD) as f:
        dset = f["data"]
        n, _, _ = dset.shape

        idxs = _idxs(n)
        # warmup
        for i in idxs[:WARMUP]:
            _ = dset[i, :, :]

        # timed
        rows = []
        for j, i in enumerate(idxs[WARMUP:WARMUP+REPS], 1):
            t0 = time.perf_counter_ns()
            _ = dset[i, :, :]
            t1 = time.perf_counter_ns()
            rows.append({"i": j, "t_total_ns": int(t1 - t0)})

    run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    out = f"logs/hdf5_{label}_slice_{run_id}.json"
    with open(out, "w") as fp:
        json.dump({"run_id": run_id, "data": rows}, fp)
    print(out)
    return out

def run_full(h5_path: str, label: str) -> str:
    with h5pyd.File(h5_path, "r", endpoint=HS_ENDPOINT, username=HS_USERNAME, password=HS_PASSWORD) as f:
        dset = f["data"]
        n, _, _ = dset.shape

        def read_all() -> int:
            t0 = time.perf_counter_ns()
            for i in range(n):
                _ = dset[i, :, :]
            t1 = time.perf_counter_ns()
            return t1 - t0

        rows = []
        for j in tqdm(range(1, REPS + 1)):
            dt = read_all()
            rows.append({"i": j, "t_total_ns": int(dt)})

    run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    out = f"logs/hdf5_{label}_full_{run_id}.json"
    with open(out, "w") as fp:
        json.dump({"run_id": run_id, "data": rows}, fp)
    print(out)
    return out

FILES = {
    "gzip": "/hdf5/gzip/all_data.h5",
    "lz4":  "/hdf5/lz4/all_data.h5",
    "zstd": "/hdf5/zstd/all_data.h5",
}

### GZIP

In [15]:
run_slice(FILES["gzip"], "gzip")

logs/hdf5_gzip_slice_20250913-175541.json


'logs/hdf5_gzip_slice_20250913-175541.json'

In [38]:
run_full(FILES["gzip"], "gzip")

100%|██████████| 3/3 [14:28<00:00, 289.44s/it]

logs/hdf5_gzip_full_20250915-134639.json


'logs/hdf5_gzip_full_20250915-134639.json'

### LZ4

In [29]:
run_slice(FILES["lz4"], "lz4")

logs/hdf5_lz4_slice_20250914-123356.json


'logs/hdf5_lz4_slice_20250914-123356.json'

In [128]:
run_full(FILES["lz4"], "lz4")

100%|██████████| 50/50 [2:06:13<00:00, 151.46s/it]  

logs/hdf5_lz4_full_20250914-095729.json


'logs/hdf5_lz4_full_20250914-095729.json'

### ZSTD

In [54]:
run_slice(FILES["zstd"], "zstd")

logs/hdf5_zstd_slice_20250914-123859.json


'logs/hdf5_zstd_slice_20250914-123859.json'

In [151]:
run_full(FILES["zstd"], "zstd")

100%|██████████| 50/50 [2:06:16<00:00, 151.54s/it]  

logs/hdf5_zstd_full_20250914-121416.json


'logs/hdf5_zstd_full_20250914-121416.json'

# ROOT

### Intro

In [ ]:
# %% ROOT run_full — one-by-one reads, resumable, single tqdm
from tqdm import tqdm
import os, json, time, datetime, tempfile, shutil, uproot

WARMUP = 10
REPS   = 50

URIS = {
    "gzip": "s3://scatterin-thesis/root/gzip.root",
    "lz4":  "s3://scatterin-thesis/root/lz4.root",
    "zstd": "s3://scatterin-thesis/root/zstd.root",
}

def _atomic_write_json(path: str, payload: dict):
    d = os.path.dirname(path) or "."
    os.makedirs(d, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=d, prefix=".ckpt_", suffix=".json")
    os.close(fd)
    with open(tmp, "w") as f:
        json.dump(payload, f)
    shutil.move(tmp, path)

def run_full(uri: str, label: str) -> str:
    ckpt_path = f"logs/temp/root_{label}_full_checkpoint.json"

    # Load checkpoint if exists
    if os.path.exists(ckpt_path):
        with open(ckpt_path, "r") as f:
            ckpt = json.load(f)
        run_id = ckpt.get("run_id") or datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        rows   = ckpt.get("data", [])
        start_rep = len(rows) + 1
    else:
        run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        rows = []
        start_rep = 1

    # Open ROOT
    f = uproot.open(uri)
    tree = f["img_tree"]
    br   = tree["data"]
    n    = tree.num_entries

    # Warmup (not recorded, no progress updates)
    with tqdm(total=3 * n, initial=0, desc=f"ROOT {label} full", unit="entry") as pbar:
        for rep in range(REPS):
            t0 = time.perf_counter_ns()
            for i in range(n):
                _ = br.array(entry_start=i, entry_stop=i+1, library="np")
                pbar.update(1)
            t1 = time.perf_counter_ns()
            rows.append({"i": rep, "t_total_ns": int(t1 - t0)})

            # Checkpoint after each rep
            _atomic_write_json(ckpt_path, {"run_id": run_id, "data": rows})

    # Save final file
    final_out = f"logs/root_{label}_full_{run_id}.json"
    _atomic_write_json(final_out, {"run_id": run_id, "data": rows})
    print(final_out)
    return final_out

### GZIP

In [80]:
run_slice(URIS["gzip"], "gzip")

logs/root_gzip_slice_20250914-130547.json


'logs/root_gzip_slice_20250914-130547.json'

In [57]:
run_full(URIS["gzip"], "gzip")

ROOT gzip full: 3885entry [21:51,  2.96entry/s]                 

logs/root_gzip_full_20250915-142929.json


'logs/root_gzip_full_20250915-142929.json'

### LZ4

In [99]:
run_slice(URIS["lz4"], "lz4")

logs/root_lz4_slice_20250914-131625.json


'logs/root_lz4_slice_20250914-131625.json'

In [69]:
run_full(URIS["lz4"], "lz4")

ROOT lz4 full: 100%|██████████| 3885/3885 [18:01<00:00,  3.59entry/s]

logs/root_lz4_full_20250915-145724.json


'logs/root_lz4_full_20250915-145724.json'

### ZSTD

In [107]:
run_slice(URIS["zstd"], "zstd")

logs/root_zstd_slice_20250914-132431.json


'logs/root_zstd_slice_20250914-132431.json'

In [83]:
run_full(URIS["zstd"], "zstd")

ROOT zstd full: 100%|██████████| 3885/3885 [18:20<00:00,  3.53entry/s]

logs/root_zstd_full_20250915-152157.json


'logs/root_zstd_full_20250915-152157.json'